# 9 — Simulated Scoring Call

What a service would do with one incoming description, minus the hosting.
Three steps, in order:

1. **Clean** the raw text with the recipe the training data went through.
2. **Gate on readability** — refuse outright if the model recognises no
   word in it.
3. **Predict at the deepest level clearing its threshold**, testing L4
   first, then L3, L2, L1.

Thresholds here are on the **raw, uncalibrated** scale.

## Setup

In [0]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [0]:
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd

from product_classifier.artifacts import load_bundle, load_cleaning_config, describe_bundle
from product_classifier.cleaning import build_cleaner
from product_classifier.scoring import score_descriptions, scoring_summary

---
## 1. Load the model

In [0]:
MODEL_DIR = "../outputs/model_v1"

clf = load_bundle(MODEL_DIR)
ID_LEVELS = clf.level_columns
NAME_LEVELS = [c.replace(" ID", " Name") for c in ID_LEVELS]

print("levels:", ID_LEVELS)
print(f"vocabulary: {len(clf.vectorizer.vocabulary_):,} features")
print(f"paths seen in training: {len(clf.valid_paths_):,}")

levels: ['Level 1 ID', 'Level 2 ID', 'Level 3 ID', 'Level 4 ID']
vocabulary: 50,000 features
paths seen in training: 2,264


In [0]:
# The ID -> name lookup saved with the model, so predictions are readable.
LOOKUP_PATH = f"{MODEL_DIR}/hierarchy_lookup.csv"
lookup = pd.read_csv(LOOKUP_PATH) if os.path.exists(LOOKUP_PATH) else None
print("lookup:", "loaded" if lookup is not None else "not found — IDs only")

lookup: loaded


### The cleaning recipe

This has to match what the training text went through, exactly. The
vocabulary was fitted on cleaned descriptions, so text cleaned any
differently looks mostly unknown — and the failure is silent, showing up
as a raised out-of-vocabulary rate rather than an error.

Notebook 4 saves the recipe into the bundle as a `DescriptionCleaner`,
word list included, so there is nothing to reproduce here. If it is
missing, rebuilding it locally is a *fallback with a caveat*: the packaged
reference lists will match, but the NLTK word list may not be the same
one training used, which shifts which leading tokens get stripped.

In [0]:
cleaner = load_cleaning_config(MODEL_DIR)

if cleaner is None:
    print("no cleaning recipe in the bundle — rebuilding it locally.")
    print("Re-save the bundle from notebook 4 with cleaning_cfg=cleaner")
    print("to remove the chance of the two drifting apart.")
    cleaner = build_cleaner()
else:
    print("cleaning recipe loaded from the bundle")

print(cleaner.describe() if hasattr(cleaner, "describe") else cleaner)

---
## 2. Thresholds

One per level, on the raw scale, tested deepest first.

Two things to know about these numbers. They are **not comparable between
levels** — a depth-1 raw score and a depth-4 raw score come from different
distributions, so a smaller number is not a lower bar. And a value that
looks low can still be strict: what matters is what share of products
clear it, not the digits.

Take them from notebook 6's threshold table rather than intuition. Omit a
level to take it out of service.

In [0]:
# depth: minimum raw prefix probability
THRESHOLDS = {
    4: 0.90,     # full path
    3: 0.90,
    2: 0.50,
    1: 0.40,
}

MIN_KNOWN_TOKENS = 2     # refuse if fewer than this many words are recognised

pd.DataFrame([{"depth": d, "level": ID_LEVELS[d - 1], "min_raw_confidence": t}
              for d, t in sorted(THRESHOLDS.items())])

,depth,level,min_raw_confidence
0,1,Level 1 ID,0.4
1,2,Level 2 ID,0.5
2,3,Level 3 ID,0.9
3,4,Level 4 ID,0.9


---
## 3. Incoming descriptions

Raw, as they would arrive — mixed case, punctuation, colours and sizes
still in. The last two are deliberately unreadable: nonsense, and an
empty string.

In [0]:
incoming = [
    "Brass Ball Valve, 1 in. NPT Threaded, Full Port",
    "nitrile exam gloves, powder-free, BLUE, medium (box of 100)",
    "SAFETY GLASSES CLEAR ANTI-FOG LENS",
    "Valve assy",
    "3/4in x 10ft galvanized steel pipe, threaded both ends",
    "zzqq wxyzzy plmnop",
    "",
]
for t in incoming:
    print(repr(t))

'Brass Ball Valve, 1 in. NPT Threaded, Full Port'
'nitrile exam gloves, powder-free, BLUE, medium (box of 100)'
'SAFETY GLASSES CLEAR ANTI-FOG LENS'
'Valve assy'
'3/4in x 10ft galvanized steel pipe, threaded both ends'
'zzqq wxyzzy plmnop'
''


---
## 4. The call

In [0]:
scored = score_descriptions(
    clf, incoming,
    thresholds=THRESHOLDS,
    cleaning_cfg=cleaner,
    min_known_tokens=MIN_KNOWN_TOKENS,
    lookup=lookup,
    name_columns=NAME_LEVELS if lookup is not None else None,
)

scored[["raw_text", "cleaned_text", "n_tokens", "n_known", "oov_rate",
        "status", "assigned_depth", "confidence"]]

,raw_text,cleaned_text,n_tokens,n_known,oov_rate,status,assigned_depth,confidence
0,"Brass Ball Valve, 1 in. NPT Threaded, Full Port",BRASS BALL VALVE 1 . NPT THREADED FULL PORT,11,8,0.272727,predicted,2,0.561252
1,"nitrile exam gloves, powder-free, BLUE, medium...",NITRILE EXAM GLOVES POWDER-FREE BOX OF 100,13,11,0.153846,predicted,4,0.997683
2,SAFETY GLASSES CLEAR ANTI-FOG LENS,SAFETY GLASSES ANTI-FOG LENS,9,8,0.111111,predicted,4,0.919273
3,Valve assy,VALVE ASSY,3,3,0.000000,refused: below every threshold,0,NaN
4,"3/4in x 10ft galvanized steel pipe, threaded b...",3/4IN 10FT GALVANIZED STEEL PIPE THREADED BOTH...,13,8,0.384615,predicted,2,0.860654
5,zzqq wxyzzy plmnop,ZZQQ WXYZZY PLMNOP,5,0,1.000000,refused: no recognisable words,0,NaN
6,,,0,0,0.000000,refused: no recognisable words,0,NaN


`status` separates the two refusals, because they need different
responses:

- **`refused: no recognisable words`** — the model has nothing to work
  with. Its confidence figure would be drawn from category frequencies
  alone, so it is not reported. This needs better source data.
- **`refused: below every threshold`** — the model read the description
  and was not sure enough at any level. This needs a human.

In [0]:
scoring_summary(scored)

,status,assigned_depth,n_items,share
0,predicted,4,2,0.285714
1,predicted,2,2,0.285714
2,refused: below every threshold,0,1,0.142857
3,refused: no recognisable words,0,2,0.285714


In [0]:
# What each product was actually assigned, down to its depth.
_cols = ["raw_text", "status", "assigned_depth", "confidence"]
_cols += NAME_LEVELS if lookup is not None else ID_LEVELS
scored[_cols]

,raw_text,status,assigned_depth,confidence,Level 1 Name,Level 2 Name,Level 3 Name,Level 4 Name
0,"Brass Ball Valve, 1 in. NPT Threaded, Full Port",predicted,2,0.561252,"Pipes, Valves, Fittings",Valves,NaN,NaN
1,"nitrile exam gloves, powder-free, BLUE, medium...",predicted,4,0.997683,Safety & Security,Personal Protective Equipment (PPE),Gloves,Disposable Gloves
2,SAFETY GLASSES CLEAR ANTI-FOG LENS,predicted,4,0.919273,Safety & Security,Personal Protective Equipment (PPE),Eyewear Accessories,Safety Glasses
3,Valve assy,refused: below every threshold,0,NaN,,,,
4,"3/4in x 10ft galvanized steel pipe, threaded b...",predicted,2,0.860654,"Pipes, Valves, Fittings",Fittings,NaN,NaN
5,zzqq wxyzzy plmnop,refused: no recognisable words,0,NaN,,,,
6,,refused: no recognisable words,0,NaN,,,,


### One call at a time

The same path for a single description, which is the shape an API handler
would take.

In [0]:
def classify(text):
    """Clean, gate, predict. Returns a plain dict."""
    row = score_descriptions(
        clf, [text], thresholds=THRESHOLDS, cleaning_cfg=cleaner,
        min_known_tokens=MIN_KNOWN_TOKENS,
        lookup=lookup, name_columns=NAME_LEVELS if lookup is not None else None,
    ).iloc[0]

    out = {"status": row["status"], "depth": int(row["assigned_depth"])}
    if row["status"] == "predicted":
        out["confidence"] = float(row["confidence"])
        out["path"] = [row[c] for c in (NAME_LEVELS if lookup is not None
                                        else ID_LEVELS)][: out["depth"]]
    else:
        out["reason"] = row["status"]
        out["oov_rate"] = float(row["oov_rate"])
    return out


for t in ["Brass Ball Valve, 1 in. NPT Threaded", "zzqq wxyzzy plmnop", ""]:
    print(f"{t!r}\n  -> {classify(t)}\n")

'Brass Ball Valve, 1 in. NPT Threaded'
  -> {'status': 'predicted', 'depth': 1, 'confidence': 0.4372554693528513, 'path': ['Pipes, Valves, Fittings']}

'zzqq wxyzzy plmnop'
  -> {'status': 'refused: no recognisable words', 'depth': 0, 'reason': 'refused: no recognisable words', 'oov_rate': 1.0}

''
  -> {'status': 'refused: no recognisable words', 'depth': 0, 'reason': 'refused: no recognisable words', 'oov_rate': 0.0}



### Where the thresholds bite

Raw confidence at every level for each description, so you can see which
threshold each one failed and by how much. Useful when a product you
expected to classify comes back refused.

In [0]:
diag = score_descriptions(
    clf, incoming, thresholds={}, cleaning_cfg=cleaner,
    min_known_tokens=MIN_KNOWN_TOKENS,
)
raw_scores = clf.predict_prefix_probabilities(
    diag["cleaned_text"], normalise=False
)

pd.concat([
    diag[["raw_text", "n_known"]],
    raw_scores[[f"L{d} Prefix Probability" for d in range(1, len(ID_LEVELS) + 1)]]
      .rename(columns={f"L{d} Prefix Probability": f"L{d}" for d in range(1, 5)}),
], axis=1).style.format({f"L{d}": "{:.4f}" for d in range(1, 5)}) if len(diag) else diag

,raw_text,n_known,L1,L2,L3,L4
0,"Brass Ball Valve, 1 in. NPT Threaded, Full Port",8,0.5613,0.5613,0.5230,0.5230
1,"nitrile exam gloves, powder-free, BLUE, medium (box of 100)",11,0.9977,0.9977,0.9977,0.9977
2,SAFETY GLASSES CLEAR ANTI-FOG LENS,8,0.9213,0.9213,0.9212,0.9193
3,Valve assy,3,0.0783,0.0783,0.0749,0.0749
4,"3/4in x 10ft galvanized steel pipe, threaded both ends",8,0.8607,0.8607,0.8607,0.8599
5,zzqq wxyzzy plmnop,0,0.0001,0.0001,0.0001,0.0000
6,,0,0.0001,0.0001,0.0001,0.0000
